## Step 1: Import Packages

In [21]:
# TensorFlow is my main library — everything runs through it
import tensorflow as tf

# I’ll build the model using Keras layers and model APIs
from keras import layers, models

## Step 2: Load the Image Datasets

In [31]:
from keras.utils import image_dataset_from_directory

# Set directories
train_dir = "../data/split/train"
val_dir = "../data/split/val"
test_dir = "../data/split/test"

# Load training data
train_ds = image_dataset_from_directory(
    train_dir,
    image_size=(224, 224),
    batch_size=32
)

# Get number of classes from the training dataset
num_classes = len(train_ds.class_names)

# Load validation data
val_ds = image_dataset_from_directory(
    val_dir,
    image_size=(224, 224),
    batch_size=32
)

# Load test data
test_ds = image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=32
)

Found 38791 files belonging to 39 classes.
Found 8318 files belonging to 39 classes.
Found 8339 files belonging to 39 classes.


## Step 3: Preprocess with MobileNetV2

In [23]:
# I am using MobileNetV2, so I need to preprocess the images in the exact format it expects.
from keras.applications.mobilenet_v2 import preprocess_input

# I will set AUTOTUNE to let TensorFlow optimize the data pipeline behind the scenes.
AUTOTUNE = tf.data.AUTOTUNE

# Here I define a function that will apply the MobileNetV2 preprocessing to each image.
def preprocess_ds(image, label):
    image = preprocess_input(image)
    return image, label

# Now I map the preprocessing function and add caching, shuffling and prefetching for performance
train_ds = train_ds.map(preprocess_ds, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.map(preprocess_ds, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

test_ds = test_ds.map(preprocess_ds, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Step 4: Load the Pretrained MobileNetV2 Base Model

In [24]:
# I will import the base MobileNetV2 model from keras.applications
from keras.applications import MobileNetV2

# I’ll load MobileNetV2 without the top (classification) layers
# input_shape must match the shape of images in my dataset
# weights="imagenet" tells it to load the pretrained weights
# include_top=False means I don't want the final Dense layers from ImageNet classification
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False,
                         weights="imagenet")

# I will freeze the base model — I don’t want to retrain its weights
base_model.trainable = False

# I can check the model summary to understand its architecture
base_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 9s 1us/step


Model: "mobilenetv2_1.00_224"

┏━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┓
┃ Layer       ┃ Output     ┃ Param ┃ Connected  ┃
┃ (type)      ┃ Shape      ┃     # ┃ to         ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━┩
│ input_layer │ (None,     │     0 │ -          │
│ (InputLaye… │ 224, 224,  │       │            │
│             │ 3)         │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ Conv1       │ (None,     │   864 │ input_lay… │
│ (Conv2D)    │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ bn_Conv1    │ (None,     │   128 │ Conv1[0][… │
│ (BatchNorm… │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ Conv1_relu  │ (None,     │     0 │ bn_Conv1[… │
│ (ReLU)      │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ expanded_c… │ (None,     │   288 │ Conv1_rel… │
│ (Depthwise… │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ expanded_c… │ (None,     │   128 │ expanded_… │
│ (BatchNorm… │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ expanded_c… │ (None,     │     0 │ expanded_… │
│ (ReLU)      │ 112, 112,  │       │            │
│             │ 32)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ expanded_c… │ (None,     │   512 │ expanded_… │
│ (Conv2D)    │ 112, 112,  │       │            │
│             │ 16)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ expanded_c… │ (None,     │    64 │ expanded_… │
│ (BatchNorm… │ 112, 112,  │       │            │
│             │ 16)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_ex… │ (None,     │ 1,536 │ expanded_… │
│ (Conv2D)    │ 112, 112,  │       │            │
│             │ 96)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_ex… │ (None,     │   384 │ block_1_e… │
│ (BatchNorm… │ 112, 112,  │       │            │
│             │ 96)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_ex… │ (None,     │     0 │ block_1_e… │
│ (ReLU)      │ 112, 112,  │       │            │
│             │ 96)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_pad │ (None,     │     0 │ block_1_e… │
│ (ZeroPaddi… │ 113, 113,  │       │            │
│             │ 96)        │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_de… │ (None, 56, │   864 │ block_1_p… │
│ (Depthwise… │ 56, 96)    │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_de… │ (None, 56, │   384 │ block_1_d… │
│ (BatchNorm… │ 56, 96)    │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_de… │ (None, 56, │     0 │ block_1_d… │
│ (ReLU)      │ 56, 96)    │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_pr… │ (None, 56, │ 2,304 │ block_1_d… │
│ (Conv2D)    │ 56, 24)    │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_1_pr… │ (None, 56, │    96 │ block_1_p… │
│ (BatchNorm… │ 56, 24)    │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_2_ex… │ (None, 56, │ 3,456 │ block_1_p… │
│ (Conv2D)    │ 56, 144)   │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_2_ex… │ (None, 56, │   576 │ block_2_e… │
│ (BatchNorm… │ 56, 144)   │       │            │
├─────────────┼────────────┼───────┼────────────┤
│ block_2_ex… │ (None, 56, │     0 │ block_2_e… │
│ (ReLU)      │ 56, 144)   │       │            │
├─────────────┼────────────┼───────┼────────────┤


 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

## Step 5: Build Custom Classifier on Top of MobileNetV2

In [32]:
# Started building the full model
model = models.Sequential([
    base_model,  # my frozen MobileNetV2

    # Global average pooling turns 7x7x1280 into 1280
    layers.GlobalAveragePooling2D(),

    # Dropout for regularization
    layers.Dropout(0.2),

    # Final classifier: softmax to predict one of the classes
    layers.Dense(num_classes, activation='softmax')

])

# Compiling the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Showing a summary of the full model
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape  ┃ Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ mobilenetv2_1.00_2… │ (None, 7, 7,  │ 2,257,… │
│ (Functional)        │ 1280)         │         │
├─────────────────────┼───────────────┼─────────┤
│ global_average_poo… │ (None, 1280)  │       0 │
│ (GlobalAveragePool… │               │         │
├─────────────────────┼───────────────┼─────────┤
│ dropout_1 (Dropout) │ (None, 1280)  │       0 │
├─────────────────────┼───────────────┼─────────┤
│ dense (Dense)       │ (None, 39)    │  49,959 │
└─────────────────────┴───────────────┴─────────┘

 Total params: 2,307,943 (8.80 MB)

 Trainable params: 49,959 (195.15 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Step 6: Compile and Train the Model

In [33]:
# First: Compile the model
# I'm using sparse_categorical_crossentropy since labels are integer-encoded (not one-hot)
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Then: Set up callbacks to avoid overfitting and save the best weights
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Finally: Train the model
# I'm training for max 15 epochs but early stopping may stop earlier
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop]
)

Epoch 1/15
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 957s 786ms/step - accuracy: 0.5025 - loss: 1.8803 - val_accuracy: 0.7674 - val_loss: 0.8257
Epoch 2/15
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 919s 757ms/step - accuracy: 0.7489 - loss: 0.8636 - val_accuracy: 0.8013 - val_loss: 0.6821
Epoch 3/15
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 926s 764ms/step - accuracy: 0.7828 - loss: 0.7265 - val_accuracy: 0.8215 - val_loss: 0.5998
Epoch 4/15
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 928s 765ms/step - accuracy: 0.7971 - loss: 0.6592 - val_accuracy: 0.8259 - val_loss: 0.5674
Epoch 5/15
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 933s 769ms/step - accuracy: 0.8117 - loss: 0.6130 - val_accuracy: 0.8307 - val_loss: 0.5508
Epoch 6/15
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 925s 763ms/step - accuracy: 0.8218 - loss: 0.5801 - val_accuracy: 0.8341 - val_loss: 0.5472
Epoch 7/15
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 964s 795ms/step - accuracy: 0.8253 - loss: 0.5598 - val_accuracy: 0.8388 - val_loss: 0.5168
Epoch 8/15
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 906s 746ms/step - ac

## Step 7: Evaluate on Test Set 

In [34]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")


261/261 ━━━━━━━━━━━━━━━━━━━━ 151s 577ms/step - accuracy: 0.8569 - loss: 0.4515
Test Accuracy: 0.8603
Test Loss: 0.4505


## Step 8 – Fine-tuning MobileNetV2

In [36]:
# 1. Unfreeze the last ~30% of MobileNetV2 layers
for layer in base_model.layers[-30:]:
    layer.trainable = True

# Ensure earlier layers remain frozen
for layer in base_model.layers[:-30]:
    layer.trainable = False

# 2. Compile with a very low learning rate
from keras import optimizers
fine_tune_lr = 1e-5
model.compile(
    optimizer=optimizers.Adam(learning_rate=fine_tune_lr),
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

# 3. Train again for a few epochs
fine_tune_epochs = 5
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=fine_tune_epochs
)

# 4. Evaluate after fine-tuning
loss, accuracy = model.evaluate(test_ds)
print(f"Test Accuracy after fine-tuning: {accuracy * 100:.2f}%")

Epoch 1/5
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 1138s 931ms/step - accuracy: 0.4019 - loss: 8.9932 - val_accuracy: 0.7288 - val_loss: 0.9537
Epoch 2/5
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 1082s 892ms/step - accuracy: 0.6995 - loss: 1.0256 - val_accuracy: 0.7909 - val_loss: 0.6985
Epoch 3/5
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 1072s 884ms/step - accuracy: 0.7721 - loss: 0.7403 - val_accuracy: 0.8253 - val_loss: 0.5543
Epoch 4/5
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 1071s 883ms/step - accuracy: 0.8184 - loss: 0.5909 - val_accuracy: 0.8492 - val_loss: 0.4753
Epoch 5/5
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 1097s 904ms/step - accuracy: 0.8472 - loss: 0.4729 - val_accuracy: 0.8667 - val_loss: 0.4222
261/261 ━━━━━━━━━━━━━━━━━━━━ 175s 671ms/step - accuracy: 0.8644 - loss: 0.4238
Test Accuracy after fine-tuning: 86.87%


### My Notes:
- I only unfreeze the last 30 layers, not the whole MobileNetV2, because too much unfreezing can cause overfitting or destroy the pretrained features.
- The learning rate is **10× smaller** than before — this is key for fine-tuning.
- I’m keeping epochs small (5) so the improvement is subtle but stable.

## Step 9: Evaluate After Fine-Tuning

In [38]:
# Evaluate the fine-tuned model on the test set
# This will tell me how my final model performs on unseen data

loss, accuracy = model.evaluate(test_ds)
print(f"Final Test Accuracy after fine-tuning: {accuracy * 100:.2f}%")
print(f"Final Test Loss after fine-tuning: {loss:.4f}")

261/261 ━━━━━━━━━━━━━━━━━━━━ 141s 541ms/step - accuracy: 0.8653 - loss: 0.4212
Final Test Accuracy after fine-tuning: 86.87%
Final Test Loss after fine-tuning: 0.4165


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Merge original training + fine-tuning metrics ---
# Accuracies
acc = history.history['accuracy'] + history_finetune.history['accuracy']
val_acc = history.history['val_accuracy'] + history_finetune.history['val_accuracy']

# Losses
loss = history.history['loss'] + history_finetune.history['loss']
val_loss = history.history['val_loss'] + history_finetune.history['val_loss']

# Epoch range for plotting
epochs = range(1, len(acc) + 1)

# --- 2. Plot Accuracy ---
plt.figure(figsize=(8,5))
plt.plot(epochs, acc, label='Training Accuracy')
plt.plot(epochs, val_acc, label='Validation Accuracy')
plt.axvline(len(history.history['accuracy']), color='gray', linestyle='--', label='Fine-tuning Start')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.savefig('accuracy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# --- 3. Plot Loss ---
plt.figure(figsize=(8,5))
plt.plot(epochs, loss, label='Training Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.axvline(len(history.history['loss']), color='gray', linestyle='--', label='Fine-tuning Start')
plt.title('Training & Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('loss_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# --- 4. Compare Test Accuracy Before vs After Fine-Tuning ---
before_acc = 0.8603  # From Step 7 evaluation
after_acc = 0.8687   # From Step 8 evaluation

labels = ['Before Fine-tuning', 'After Fine-tuning']
values = [before_acc, after_acc]

plt.figure(figsize=(6,5))
bars = plt.bar(labels, values, color=['skyblue', 'lightgreen'])
plt.ylabel('Test Accuracy')
plt.title('Test Accuracy Improvement')
plt.ylim(0, 1)

# Add value labels on bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.005, f"{yval*100:.2f}%", ha='center', fontsize=10)

plt.savefig('test_accuracy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
